# Brain — Multi-Agent System
**Google Colab notebook** — clone the repo, install deps, mount Drive for persistence, run the brain.

Recommended runtime: **L4 GPU** (Colab Pro) or **T4** (free tier).

If you want local Ollama models instead of Gemini, jump to the **Ollama (optional)** section.

## 1 — Mount Google Drive (data persistence)
ChromaDB and the episode SQLite database will be stored on Drive so they survive session restarts.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib

# All persistent data goes here — survives session restarts
DRIVE_DATA = '/content/drive/MyDrive/Brain/data'
pathlib.Path(DRIVE_DATA).mkdir(parents=True, exist_ok=True)

# Symlink so the code always finds data/ in the repo root
REPO_DATA = '/content/Brain/data'
if os.path.islink(REPO_DATA):
    os.remove(REPO_DATA)
# (symlink created after clone in step 3)
print(f'Drive mounted. Data will persist at: {DRIVE_DATA}')

Mounted at /content/drive
Drive mounted. Data will persist at: /content/drive/MyDrive/Brain/data


## 2 — Clone the repo

In [2]:
import os

REPO_URL = 'https://github.com/Seydifa/Brain.git'
REPO_DIR = '/content/Brain'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

Cloning into '/content/Brain'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 32 (delta 2), reused 32 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 26.26 KiB | 26.26 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/Brain
Working directory: /content/Brain


## 3 — Link Drive data directory

In [3]:
import os

REPO_DATA = '/content/Brain/data'
DRIVE_DATA = '/content/drive/MyDrive/Brain/data'

# Remove any existing data/ folder in the repo
if os.path.exists(REPO_DATA) and not os.path.islink(REPO_DATA):
    import shutil
    shutil.rmtree(REPO_DATA)
if os.path.islink(REPO_DATA):
    os.remove(REPO_DATA)

os.symlink(DRIVE_DATA, REPO_DATA)
print(f'data/ -> {DRIVE_DATA}')

data/ -> /content/drive/MyDrive/Brain/data


## 4 — Install dependencies

In [4]:
!pip install -q \
    langgraph \
    langgraph-checkpoint-sqlite \
    langchain-google-genai \
    langchain-chroma \
    langchain-ollama \
    ddgs \
    httpx \
    python-dotenv

print('Dependencies installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 81.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 150.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 131.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

## 5 — API key
Paste your Gemini API key below (get one free at https://aistudio.google.com/apikey).

In [5]:
import os
from google.colab import userdata

# Option A: store key in Colab Secrets (Colab sidebar > key icon) — recommended
try:
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('Key loaded from Colab Secrets.')
except Exception:
    # Option B: paste directly (less secure)
    os.environ['GOOGLE_API_KEY'] = 'YOUR_GEMINI_API_KEY_HERE'
    print('Key set inline.')

# Write .env so python-dotenv picks it up too
with open('/content/Brain/.env', 'w') as f:
    f.write(f"GOOGLE_API_KEY={os.environ['GOOGLE_API_KEY']}\n")

Key set inline.


## 6 — (Optional) Ollama with local models
Skip this section if you want to use Gemini API.  
On L4 GPU: `llama3.2:3b` responds in ~2 s per call. On T4: ~5 s.

> **No file patching** — the override lives entirely in memory via `sys.modules` injection, so the repo stays clean.

In [1]:
# Install prerequisites and Ollama
!apt-get install -y -q zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, shutil, os

# Reload PATH so the newly installed binary is found
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')

ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
if not os.path.isfile(ollama_bin):
    raise RuntimeError(f'Ollama not found at {ollama_bin} — install may have failed above.')

subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# Pull a model — must support tool calling for the ReAct search agent:
# 'llama3.2:3b'    → 2 GB VRAM, fast, full tool support  ← recommended
# 'llama3.1:8b'    → 8 GB VRAM, best quality, full tool support
# 'mistral:7b'     → 8 GB VRAM, strong reasoning, tool support
OLLAMA_MODEL = 'llama3.2:3b'
!ollama pull {OLLAMA_MODEL}
print(f'Model {OLLAMA_MODEL} ready.')

Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Model llama3.2:3b ready.


In [ ]:
# Override langchain_google_genai with Ollama — no file patching needed
import sys
from types import ModuleType
from langchain_ollama import ChatOllama, OllamaEmbeddings

OLLAMA_MODEL = 'llama3.2:3b'   # must support tool calling
EMBED_MODEL  = 'nomic-embed-text'

class _OllamaLLM(ChatOllama):
    """Drop-in for ChatGoogleGenerativeAI — ignores Gemini model name."""
    def __init__(self, model=None, temperature=0, **kwargs):
        kwargs.pop('convert_system_message_to_human', None)
        super().__init__(model=OLLAMA_MODEL, temperature=temperature, **kwargs)

class _OllamaEmbeddings(OllamaEmbeddings):
    """Drop-in for GoogleGenerativeAIEmbeddings."""
    def __init__(self, model=None, **kwargs):
        super().__init__(model=EMBED_MODEL, **kwargs)

# Inject before any brain modules are imported
fake = ModuleType('langchain_google_genai')
fake.ChatGoogleGenerativeAI       = _OllamaLLM
fake.GoogleGenerativeAIEmbeddings = _OllamaEmbeddings
sys.modules['langchain_google_genai'] = fake

print(f'Ollama override active')
print(f'  LLM        : {OLLAMA_MODEL}')
print(f'  Embeddings : {EMBED_MODEL}')
print(f'  Repo files : unchanged (no patching)')

Repo root: /content/Brain
Restored original files from git.
patched memory/store.py
patched memory/agent.py
patched agents/qa_agent.py
patched agents/orchestrator.py
patched agents/search_agent.py
patched agents/goal_evaluator.py
patched agents/search_validator.py
Ollama patch complete.


## 7 — Load the graph

In [13]:
import sys

# Clear all Brain module caches so patched files are re-imported fresh
brain_mods = [k for k in sys.modules if k.startswith(('core', 'agents', 'memory'))]
for m in brain_mods:
    del sys.modules[m]

sys.path.insert(0, '/content/Brain')

import uuid
from dotenv import load_dotenv
load_dotenv()

from core.graph import get_graph

graph = get_graph()
thread_id = str(uuid.uuid4())
config = {'configurable': {'thread_id': thread_id}}

EMPTY_STATE = {
    'goal': '',
    'messages': [],
    'response': '',
    'status': 'empty',
    'oriented_context': {},
    'reasoning_trace': [],
    'retry_count': 0,
    'search_valid': False,
    'search_feedback': '',
    'qa_draft': '',
    'qa_approved': False,
    'qa_feedback': '',
    'qa_attempts': 0,
    'needs_clarification': False,
    'clarification_questions': [],
}

print('Graph loaded. Ready to ask questions.')

Graph loaded. Ready to ask questions.


## 8 — Ask the Brain
Change `goal` and re-run this cell for each question.

In [14]:
import time

goal = "What caused World War 2?"  # <-- change this

state = {**EMPTY_STATE, 'goal': goal}

# Retry on rate limit (Gemini free tier)
for attempt in range(4):
    try:
        result = graph.invoke(state, config=config)
        break
    except Exception as e:
        if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
            wait = 15 * (attempt + 1)
            print(f'Rate limit — retrying in {wait}s...')
            time.sleep(wait)
        else:
            raise

# Handle clarification request
if result.get('needs_clarification'):
    print('Brain needs clarification:')
    for q in result.get('clarification_questions', []):
        print(f'  - {q}')
else:
    print('=== BRAIN RESPONSE ===')
    print(result.get('response', '(no response)'))

Brain needs clarification:
  - Was World War 2 an isolated event or part of a larger global conflict?
  - Were there any specific events or factors that led to the outbreak of World War 2?
  - Can you find information on the role of key leaders or countries in causing the war?


## 9 — Debug: inspect reasoning trace

In [15]:
ctx   = result.get('oriented_context', {})
trace = result.get('reasoning_trace', [])

print(f"Turn type  : {ctx.get('turn_type', '?')}")
print(f"Coverage   : {ctx.get('coverage', '?')}")
print(f"Confidence : {ctx.get('knowledge_confidence', 0):.2f}")
print(f"Episode    : {ctx.get('current_episode_id', '?')}")
print()
print('Reasoning trace:')
for i, step in enumerate(trace, 1):
    print(f'  {i}. {step}')

Turn type  : follow_up
Coverage   : none
Confidence : 0.00
Episode    : ep_9b0c304c_20260413003202

Reasoning trace:
  1. classified as follow_up | coverage=none | parent=ep_9b0c304c_20260413003115
  2. search attempt 1 | coverage=none | query=fresh
  3. search valid=False retry 1 | The search result does not provide a direct answer to the us
  4. search attempt 2 | coverage=none | query=feedback-reformulated
  5. search valid=False retry 2 | The search result does not provide a direct answer to the us
  6. search attempt 3 | coverage=none | query=feedback-reformulated
  7. search valid=False retry 3 | The search result does not provide a direct answer to the us


## 10 — View episode history

In [18]:
from memory.episodes import get_recent
import json

episodes = get_recent(10)
for ep in episodes:
    flags = json.loads(ep.get('flags') or '[]')
    print(f"[{ep['id']}]")
    print(f"  Request  : {ep['user_request'][:80]}")
    print(f"  Type     : {ep['turn_type']}")
    print(f"  Flags    : {flags}")
    print(f"  Response : {str(ep.get('chosen_response', ''))[:100]}...")
    print()

[ep_9b0c304c_20260413003202]
  Request  : What caused World War 2?
  Type     : follow_up
  Flags    : ['in_progress']
  Response : ...

[ep_9b0c304c_20260413003115]
  Request  : What caused World War 2?
  Type     : follow_up
  Flags    : ['in_progress']
  Response : ...

[ep_9b0c304c_20260413002751]
  Request  : What caused World War 2?
  Type     : new_topic
  Flags    : ['in_progress']
  Response : ...



In [ ]:
## 11 — Long stateful conversation (12 turns)
A single thread running through **modern physics**: general relativity → gravitational waves → black holes → dark matter → synthesis.

Tests:
- **Memory continuity** — follow-up classification across many turns
- **Coverage accumulation** — knowledge reuse as the topic deepens
- **Topic pivots** — correct `new_topic` re-classification
- **Synthesis** — final turn draws on everything stored in ChromaDB


[1/5] Explain how the human immune system works.
  Outcome   : ANSWERED
  Turn type : new_topic
  Coverage  : full  (confidence 0.84)
  Time      : 10.3s
  Response  :
    **How the Human Immune System Works**  The human immune system is a complex
    network of cells, tissues, and organs that work together to protect the body
    against pathogens, such as bacteria, viruses, and other foreign substances.  ###
    Recognition  1. The immune system recognizes the presence of a pathogen through
    various mechanisms, including:         * Pattern recognition receptors (PRRs) on
    the surface of immune cells that recognize specific patterns on the surface of
    pathogens.         * Antigen-presenting cells (APCs) that engulf and process
    pathogens, then display pieces of them on their surface for other immune cells
    to recognize.  ### Activation  1. When an immune cell recognizes a pathogen, it
    becomes activated and begins to proliferate and differentiate into specialized
  

In [ ]:
import time, textwrap
from IPython.display import display, Markdown

CONVERSATION = [
    # --- Thread 1: General Relativity (turns 1-5) ---
    "Explain the theory of general relativity and what it changed about our understanding of gravity.",
    "How does general relativity differ from Newton's theory of gravity and from special relativity?",
    "What are gravitational waves and how were they first detected?",
    "Who were the key scientists and institutions behind the LIGO gravitational wave detection?",
    "How does GPS rely on both special and general relativity to stay accurate to within metres?",
    # --- Thread 2: Black Holes (turns 6-9) ---
    "What is a black hole and how do they form from dying stars?",
    "What happens at the event horizon of a black hole — can anything escape?",
    "What is Hawking radiation and why does it suggest black holes eventually evaporate?",
    "How are supermassive black holes connected to galaxy formation and active galactic nuclei?",
    # --- Thread 3: Dark Universe (turns 10-11) ---
    "What is dark matter? What observational evidence do we have for its existence?",
    "What is dark energy and how does it explain the accelerating expansion of the universe?",
    # --- Turn 12: Synthesis ---
    "Summarise the key unsolved mysteries in modern physics that we have discussed across all our conversations.",
]

# Reuse the same thread so LangGraph checkpointing + episode diary carry memory
conv_config = {'configurable': {'thread_id': thread_id}}
all_results = []
total_t0 = time.time()

for idx, goal in enumerate(CONVERSATION, 1):
    t0 = time.time()
    state = {**EMPTY_STATE, 'goal': goal}

    for attempt in range(4):
        try:
            r = graph.invoke(state, config=conv_config)
            break
        except Exception as e:
            if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                wait = 15 * (attempt + 1)
                display(Markdown(f'> ⏳ Rate limit — retrying in {wait}s…'))
                time.sleep(wait)
            else:
                raise

    elapsed   = time.time() - t0
    ctx_r     = r.get('oriented_context', {})
    turn_type = ctx_r.get('turn_type', '?')
    coverage  = ctx_r.get('coverage', '?')
    conf      = ctx_r.get('knowledge_confidence', 0)

    if r.get('needs_clarification'):
        outcome = 'CLARIFICATION'
        body    = '\n'.join(f'- {q}' for q in r.get('clarification_questions', []))
    else:
        outcome = 'ANSWERED'
        body    = r.get('response', '*(no response)*')

    badge = {'ANSWERED': '🟢', 'CLARIFICATION': '🟡'}.get(outcome, '⚪')
    display(Markdown(f"""
---
### Turn {idx} / {len(CONVERSATION)} &nbsp; {badge} {outcome}

**Q:** *{goal}*

{body}

> `turn={turn_type}` &nbsp;|&nbsp; `coverage={coverage}` &nbsp;|&nbsp; `confidence={conf:.2f}` &nbsp;|&nbsp; `{elapsed:.1f}s`
"""))

    all_results.append(dict(
        q=idx, goal=goal[:60]+'…' if len(goal)>60 else goal,
        outcome=outcome, turn_type=turn_type,
        coverage=coverage, conf=f'{conf:.2f}', time_s=f'{elapsed:.1f}',
    ))

total_elapsed = time.time() - total_t0

rows = '\n'.join(
    f"| {row['q']:>2} | {row['outcome']:<15} | {row['turn_type']:<12} | {row['coverage']:<9} | {row['conf']:>5} | {row['time_s']:>5}s |"
    for row in all_results
)
display(Markdown(f"""
---
## Summary — {len(CONVERSATION)} turns &nbsp; ⏱ {total_elapsed:.0f}s total

| # | Outcome | Turn type | Coverage | Conf | Time |
|--:|---------|-----------|----------|-----:|------|
{rows}
"""))

 #  Outcome         Type         Coverage    Conf   Time  Question
--------------------------------------------------------------------------------
 1  ANSWERED        new_topic    full        0.84  10.3s  Explain how the human immune system works.
 2  ANSWERED        new_topic    full        0.69   5.2s  What is the difference between innate and adaptive immu…
 3  ANSWERED        new_topic    full        0.68   9.3s  How do vaccines train the adaptive immune system?
 4  ANSWERED        new_topic    full        0.61   6.3s  What makes mRNA vaccines like Pfizer-BioNTech different…
 5  CLARIFICATION   new_topic    partial     0.46   4.9s  What is quantum entanglement and why did Einstein call …
